# Build a RAG Pipeline — Solution

## Setup

In [ ]:
import torch
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

emb_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

gen_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForCausalLM.from_pretrained(gen_model_name, dtype=torch.float16)

device = "cuda" if torch.cuda.is_available() else "cpu"
gen_model = gen_model.to(device)
print(f"Generation model on {device}")

## Task 1: Explore the Data

In [ ]:
# Combine both splits into one dataset
dataset = load_dataset("PhilSad/SCP-Wiki-Dataset", split="train+test")
df = pd.DataFrame(dataset)
print(f"{len(df)} entries loaded")
df.head()

In [ ]:
# Check content lengths
df["content_len"] = df["content"].str.len()
print(df["content_len"].describe())
print()
print(f"Entries over 10k chars: {(df['content_len'] > 10000).sum()}")
print(f"Entries under 1k chars: {(df['content_len'] < 1000).sum()}")

In [ ]:
# Look at a short entry and a long entry
shortest = df.loc[df["content_len"].idxmin()]
print(f"=== Shortest entry ({shortest['content_len']} chars) ===")
print(shortest["content"][:500])
print()

longest = df.loc[df["content_len"].idxmax()]
print(f"=== Longest entry ({longest['content_len']} chars) ===")
print(longest["content"][:500])

## Task 2: Chunk the Corpus

In [ ]:
def chunk_corpus(df):
    """Take the SCP dataframe and return a list of text chunks for embedding."""
    chunks = []
    max_chars = 1000
    
    for _, row in df.iterrows():
        text = row["content"]
        
        # Split on double newlines (section breaks in SCP format)
        sections = text.split("\n\n")
        
        current_chunk = ""
        for section in sections:
            # If a single section exceeds max_chars, split further on newlines
            if len(section) > max_chars:
                if current_chunk.strip():
                    chunks.append(current_chunk.strip())
                    current_chunk = ""
                lines = section.split("\n")
                for line in lines:
                    if len(current_chunk) + len(line) > max_chars and current_chunk:
                        chunks.append(current_chunk.strip())
                        current_chunk = line
                    else:
                        current_chunk += "\n" + line if current_chunk else line
            elif len(current_chunk) + len(section) > max_chars and current_chunk:
                chunks.append(current_chunk.strip())
                current_chunk = section
            else:
                current_chunk += "\n\n" + section if current_chunk else section
        
        if current_chunk.strip():
            chunks.append(current_chunk.strip())
    
    # Filter out empty chunks
    return [c for c in chunks if c]

In [ ]:
chunks = chunk_corpus(df)
print(f"{len(chunks)} chunks created")

chunk_lens = [len(c) for c in chunks]
print(f"Min: {min(chunk_lens)}, Max: {max(chunk_lens)}, Mean: {sum(chunk_lens)/len(chunk_lens):.0f}")

## Task 3: Embed the Chunks

In [ ]:
chunk_embeddings = emb_model.encode(chunks, convert_to_tensor=True, show_progress_bar=True)

## Task 4: Write Retrieval

In [ ]:
def retrieve(query, chunk_embeddings, chunks, emb_model, k=3):
    """Return the top-k most similar chunks to the query."""
    query_embedding = emb_model.encode(query, convert_to_tensor=True)
    similarities = emb_model.similarity(query_embedding, chunk_embeddings).squeeze()
    top_k = similarities.argsort(descending=True)[:k]
    
    results = []
    for idx in top_k:
        results.append({
            "text": chunks[idx],
            "score": similarities[idx].item()
        })
    return results

In [ ]:
results = retrieve("What SCP is a staircase?", chunk_embeddings, chunks, emb_model)

for r in results:
    print(f"[{r['score']:.3f}]")
    print(r["text"][:200])
    print()

In [ ]:
results = retrieve("Which SCP can destroy the world?", chunk_embeddings, chunks, emb_model)

for r in results:
    print(f"[{r['score']:.3f}]")
    print(r["text"][:200])
    print()

## Task 5: Build the RAG Pipeline

In [ ]:
def ask(question, chunk_embeddings, chunks, emb_model, gen_model, gen_tokenizer, k=3):
    """Retrieve context and generate an answer to the question."""
    results = retrieve(question, chunk_embeddings, chunks, emb_model, k=k)
    
    context = "\n\n".join([r["text"] for r in results])
    
    prompt = f"""Answer the following question using only the context provided. If the context doesn't contain enough information, say so.

Context:
{context}

Question: {question}"""
    
    messages = [{"role": "user", "content": prompt}]
    inputs = gen_tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(device)
    
    gen_model.eval()
    with torch.no_grad():
        output = gen_model.generate(**inputs, max_new_tokens=256, do_sample=False)
    
    return gen_tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

In [ ]:
print(ask("What SCP is a staircase that goes down forever?", chunk_embeddings, chunks, emb_model, gen_model, gen_tokenizer))

In [ ]:
# Compare: same question without RAG
question = "What SCP is a staircase that goes down forever?"

messages = [{"role": "user", "content": question}]
inputs = gen_tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(device)

gen_model.eval()
with torch.no_grad():
    output = gen_model.generate(**inputs, max_new_tokens=256, do_sample=False)

print(gen_tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

In [ ]:
# RAG helps: domain-specific question
print("=== With RAG ===")
print(ask("What is SCP-173?", chunk_embeddings, chunks, emb_model, gen_model, gen_tokenizer))
print()

# RAG doesn't help: out-of-domain question
print("=== Out of domain ===")
print(ask("What is the capital of France?", chunk_embeddings, chunks, emb_model, gen_model, gen_tokenizer))